In [0]:
spark

# Linear Regression

In [0]:
from pyspark.sql.types import * 
df=spark.read.format('csv').option('inferschema',True).option('header',True).load('/Volumes/workspace/default/sampledataset/csvfiles/big data dataset.csv')

df.display()
df.printSchema()
print(df.columns)
df.show()

In [0]:
df.printSchema()

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

feature_columns = [
    'stress_level', 
    'active_minutes', 
    'workouts_per_week', 
    'heart_rate_resting'
]
target_column = 'sleep_quality'


cleaned_df = df.select(feature_columns + [target_column]).na.drop()


assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
assembled_df = assembler.transform(cleaned_df)

train_data, test_data = assembled_df.randomSplit([0.8, 0.2], seed=42)

lr_reg = LinearRegression(
    featuresCol="features", 
    labelCol=target_column,
    regParam=0.1,        
    elasticNetParam=0.5  
)
lr_reg_model = lr_reg.fit(train_data)


predictions_reg = lr_reg_model.transform(test_data)


print("\nSample Predictions vs Actual (With Regularization):")

predictions_reg.select("stress_level", "active_minutes", target_column, "prediction").show(10)


rmse_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="rmse")
r2_evaluator = RegressionEvaluator(labelCol=target_column, predictionCol="prediction", metricName="r2")

print("-" * 30)
print(f"RMSE: {rmse_evaluator.evaluate(predictions_reg):.4f}")
print(f"R2:   {r2_evaluator.evaluate(predictions_reg):.4f}")
print("-" * 30)
import matplotlib.pyplot as plt
import pandas as pd

predictions=predictions_reg
pandas_df = predictions.select(target_column, "prediction").toPandas()


plt.figure(figsize=(8, 6))

plt.scatter(pandas_df[target_column], pandas_df["prediction"], alpha=0.5, color='royalblue', label='Predictions')


min_val = min(pandas_df[target_column].min(), pandas_df["prediction"].min())
max_val = max(pandas_df[target_column].max(), pandas_df["prediction"].max())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label="Perfect Fit (y=x)")


plt.xlabel("Actual Sleep Quality (1-10)")
plt.ylabel("Predicted Sleep Quality (1-10)")
plt.title("Actual vs. Predicted Sleep Quality")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.show()